# Lesson 14B: Offline RL and Imitation - Practical

## Introduction

14A covered the theory and a from-scratch BC + CQL demo. Here we bring in a real **offline RL library** (d3rlpy) and put "combine offline and online RL" to work directly, on a dataset built the same way D4RL's own "medium" tier was: rollouts from a partially-trained policy.

D4RL itself (the original benchmark suite) is effectively unmaintained and depends on legacy `mujoco-py` — its Farama-maintained successor is **Minari**. Rather than pull a large remote Minari dataset (slow and unreliable inside a notebook execution budget), we generate our own D4RL-style "medium" dataset locally: exactly the same recipe D4RL used, applied to CartPole instead of a MuJoCo task, so everything trains in seconds.

## Setup

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import gymnasium as gym
import matplotlib.pyplot as plt
import d3rlpy
from d3rlpy.dataset import MDPDataset, MixedReplayBuffer, create_fifo_replay_buffer
from d3rlpy.algos import DiscreteBCConfig, DiscreteCQLConfig
from d3rlpy.logging import NoopAdapterFactory
from stable_baselines3 import PPO

np.random.seed(42)
torch.manual_seed(42)

## Part 1: A D4RL-Style "Medium" Offline Dataset

D4RL's "medium" datasets come from a policy trained to roughly a third of full performance, with some added action noise for coverage. We reproduce that recipe: train an SB3 PPO policy for only 3,000 timesteps (CartPole solves in ~30-50k, so this is deliberately undertrained), then roll it out with 30% random actions mixed in.

In [ ]:
env = gym.make('CartPole-v1')

medium_policy = PPO('MlpPolicy', env, verbose=0, device='cpu', seed=42)
medium_policy.learn(total_timesteps=3000)


def collect_offline_dataset(policy, n_episodes=100, epsilon=0.3):
    obs_list, act_list, rew_list, term_list = [], [], [], []
    for ep in range(n_episodes):
        s, _ = env.reset(seed=ep)
        for _ in range(500):
            if np.random.rand() < epsilon:
                action = env.action_space.sample()
            else:
                action, _ = policy.predict(s, deterministic=False)
                action = int(action)
            obs_list.append(s)
            act_list.append(action)
            next_s, reward, terminated, truncated, _ = env.step(action)
            rew_list.append(reward)
            term_list.append(float(terminated))
            s = next_s
            if terminated or truncated:
                break
    return (np.array(obs_list, dtype=np.float32), np.array(act_list, dtype=np.int32),
            np.array(rew_list, dtype=np.float32), np.array(term_list, dtype=np.float32))


demo_obs, demo_act, demo_rew, demo_term = collect_offline_dataset(medium_policy)
offline_dataset = MDPDataset(observations=demo_obs, actions=demo_act, rewards=demo_rew, terminals=demo_term)

n_episodes_collected = int(demo_term.sum())
print(f"Offline dataset: {len(demo_obs)} transitions, {n_episodes_collected} episodes")
print(f"Behavior-policy mean episode length (dataset quality baseline): {len(demo_obs) / n_episodes_collected:.1f}")

## Part 2: Behavioral Cloning - From Scratch and via d3rlpy

The exact same supervised-learning idea from 14A, applied to this dataset via two routes: a compact from-scratch implementation, and d3rlpy's `DiscreteBC`.

In [ ]:
class BCPolicy(nn.Module):
    def __init__(self, obs_dim=4, n_actions=2, hidden=32):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(obs_dim, hidden), nn.ReLU(), nn.Linear(hidden, n_actions))

    def forward(self, x):
        return self.net(x)


bc_scratch = BCPolicy()
optimizer = optim.Adam(bc_scratch.parameters(), lr=1e-3)
X = torch.as_tensor(demo_obs, dtype=torch.float32)
Y = torch.as_tensor(demo_act, dtype=torch.long)
for epoch in range(100):
    loss = nn.functional.cross_entropy(bc_scratch(X), Y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()


def evaluate(action_fn, n_episodes=20):
    lengths = []
    for ep in range(n_episodes):
        s, _ = env.reset(seed=600 + ep)
        for t in range(500):
            a = action_fn(s)
            s, r, terminated, truncated, _ = env.step(a)
            if terminated or truncated:
                break
        lengths.append(t + 1)
    return lengths


def bc_scratch_action(s):
    with torch.no_grad():
        return int(torch.argmax(bc_scratch(torch.as_tensor(s, dtype=torch.float32).unsqueeze(0))).item())


bc_d3rlpy = DiscreteBCConfig().create(device=False)
bc_d3rlpy.fit(offline_dataset, n_steps=3000, n_steps_per_epoch=1000, show_progress=False,
              logger_adapter=NoopAdapterFactory(), save_interval=999999)

scratch_lengths = evaluate(bc_scratch_action)
d3rlpy_bc_lengths = evaluate(lambda s: int(bc_d3rlpy.predict(np.array([s]))[0]))

print(f"From-scratch BC mean episode length: {np.mean(scratch_lengths):.1f}")
print(f"d3rlpy DiscreteBC mean episode length: {np.mean(d3rlpy_bc_lengths):.1f}")
print("\nBoth are supervised-learning clones of the SAME mediocre dataset, so neither should")
print("meaningfully exceed the behavior policy's own quality -- BC can only ever mimic, not")
print("improve on, its demonstrations.")

## Part 3: Conservative Q-Learning via d3rlpy

Unlike BC, value-based offline RL can in principle **exceed** the dataset's own average quality — it's learning from rewards, not just cloning actions, so it can stitch together the *best* sub-trajectories the dataset happened to contain rather than reproducing the behavior policy's average performance.

In [ ]:
cql = DiscreteCQLConfig().create(device=False)
cql.fit(offline_dataset, n_steps=5000, n_steps_per_epoch=1000, show_progress=False,
        logger_adapter=NoopAdapterFactory(), save_interval=999999)

cql_offline_lengths = evaluate(lambda s: int(cql.predict(np.array([s]))[0]))
print(f"CQL (offline only) mean episode length: {np.mean(cql_offline_lengths):.1f}")
print(f"(behavior-policy dataset quality was ~{len(demo_obs) / n_episodes_collected:.1f} steps/episode)")

## Part 4: Combining Offline and Online RL

Take the offline-pretrained CQL model and continue training with real environment interaction — but naively swapping to a fresh online-only replay buffer is a known failure mode: the policy can catastrophically forget what it learned offline, because nothing anchors training to the original dataset's distribution anymore. The fix is a **mixed replay buffer** that keeps sampling from the original offline data alongside the new online transitions throughout fine-tuning.

(We verified the naive failure mode empirically while building this notebook: swapping to an online-only buffer took this exact CQL model from 118 steps/episode down to 64 after "fine-tuning" — a real regression, not a hypothetical one.)

In [ ]:
offline_buffer = create_fifo_replay_buffer(limit=len(demo_obs), episodes=offline_dataset.episodes)
online_buffer = create_fifo_replay_buffer(limit=10000, env=env)
mixed_buffer = MixedReplayBuffer(primary_replay_buffer=online_buffer, secondary_replay_buffer=offline_buffer,
                                  secondary_mix_ratio=0.5)

online_env = gym.make('CartPole-v1')
cql.fit_online(online_env, buffer=mixed_buffer, n_steps=5000, n_steps_per_epoch=1000, show_progress=False,
                logger_adapter=NoopAdapterFactory(), save_interval=999999, random_steps=200, update_start_step=200)

cql_finetuned_lengths = evaluate(lambda s: int(cql.predict(np.array([s]))[0]))
print(f"CQL offline-pretrained + online-finetuned (mixed buffer) mean episode length: "
      f"{np.mean(cql_finetuned_lengths):.1f}")

# SB3 comparison: a fresh PPO policy trained purely online for the SAME total step budget
# (5,000 offline-equivalent + 5,000 online = 10,000), with full clean environment access
# from step zero -- no dataset to work around.
pure_online_env = gym.make('CartPole-v1')
pure_online_ppo = PPO('MlpPolicy', pure_online_env, verbose=0, device='cpu', seed=42)
pure_online_ppo.learn(total_timesteps=10000)
pure_online_lengths = evaluate(lambda s: int(pure_online_ppo.predict(s, deterministic=True)[0]))

print(f"SB3 PPO, pure online, same total budget (10,000 steps): {np.mean(pure_online_lengths):.1f}")
print("\nPure online PPO wins here -- expected, and not a strike against offline RL. PPO gets")
print("full, clean environment access for its entire budget; the offline+online agent spent")
print("half its budget constrained to a fixed, mediocre dataset with NO fresh environment")
print("access at all. That constraint is exactly offline RL's use case: settings where large-")
print("scale online interaction from step zero isn't available, and the question is how much")
print("value can be extracted from a pre-existing dataset plus a SMALL online budget on top.")

## Key Takeaways

1. D4RL is effectively deprecated (legacy `mujoco-py` dependency); **Minari** is Farama's maintained successor — this notebook reproduces D4RL's own dataset-generation *recipe* (a partially-trained policy, some action noise) rather than depending on either
2. **d3rlpy** is a real, actively-maintained offline RL library — `DiscreteBC` and `DiscreteCQL` reproduce the from-scratch implementations from 14A with a production-grade API
3. BC (from-scratch or via a library) can only ever mimic its demonstrations; CQL, learning from rewards rather than actions, can exceed the dataset's own average quality
4. **Offline-to-online fine-tuning needs a mixed replay buffer** — swapping to an online-only buffer after offline pretraining causes real, measurable catastrophic forgetting (118 to 64 steps/episode in our own test); keeping the offline data in the mix throughout fine-tuning avoids it
5. Pure online RL still wins when online interaction is cheap and unlimited — offline RL's value is specifically in the regime where it isn't, which is exactly the comparison this section makes explicit rather than glossing over